# El que sabreu fer al gener

**Optativa d'Aprenentatge automàtic · sessió 2**

Ahir vau fer condicionals: `if`, `elif`, `else`. Avui us ensenyo on porta això.

No cal que entengueu el codi d'aquest quadern. Mireu els resultats i la última part,
que és on hi ha la gràcia. Al gener això ho sabreu escriure vosaltres.

## 1. Les dades

Farem servir un conjunt de dades clàssic: **150 flors d'iris**, de tres espècies
diferents. De cada flor tenim quatre mesures en centímetres: llargada i amplada del
sèpal, i llargada i amplada del pètal.

La pregunta és senzilla: **si et dono les quatre mesures d'una flor, em sabries dir
de quina espècie és?**

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
dades = iris.frame.copy()
dades["especie"] = iris.target_names[iris.target]
dades = dades.rename(columns={
    "sepal length (cm)": "sepal_llarg",
    "sepal width (cm)": "sepal_ample",
    "petal length (cm)": "petal_llarg",
    "petal width (cm)": "petal_ample",
})

print("Files:", len(dades))
print(dades["especie"].value_counts().to_string())
dades.head()

## 2. Mirem-les abans de fer res

Primera regla de tot això: **mira les dades abans de tocar-les**. Un gràfic diu més
que qualsevol resum.

Posem la llargada del pètal a l'eix horitzontal i l'amplada a l'eix vertical, i pintem
cada espècie d'un color.

In [ ]:
import matplotlib.pyplot as plt

colors = {"setosa": "tab:blue", "versicolor": "tab:orange", "virginica": "tab:green"}

plt.figure(figsize=(8, 5))
for especie, grup in dades.groupby("especie"):
    plt.scatter(grup["petal_llarg"], grup["petal_ample"],
                label=especie, color=colors[especie], s=40, alpha=0.8)

plt.xlabel("Llargada del pètal (cm)")
plt.ylabel("Amplada del pètal (cm)")
plt.title("Les tres espècies, segons el pètal")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. El repte: escriu-ho tu amb el que saps

Mirant el gràfic, ja veus que els grups se separen bastant bé. Amb els condicionals
d'ahir, series capaç d'escriure les regles?

Alguna cosa així:

> Si el pètal és molt curt → *setosa*.
> Si no, i el pètal és estret → *versicolor*.
> Si no → *virginica*.

Provem-ho. Això és **només `if`, `elif` i `else`**: el que vau fer ahir.

In [ ]:
def classifica_a_ma(petal_llarg, petal_ample):
    if petal_llarg < 2.5:
        return "setosa"
    elif petal_ample < 1.75:
        return "versicolor"
    else:
        return "virginica"


encerts = 0
for _, flor in dades.iterrows():
    predit = classifica_a_ma(flor["petal_llarg"], flor["petal_ample"])
    if predit == flor["especie"]:
        encerts += 1

print(f"Encerts: {encerts} de {len(dades)}")
print(f"Precisió: {encerts / len(dades):.1%}")

Un 96 % amb tres línies de condicionals. Gens malament.

Però fixa't en què has hagut de fer: **mirar el gràfic, pensar, i triar els números
2.5 i 1.75 a ull**. Has trigat uns minuts i només tenies 4 columnes i 150 files.

Ara imagina que en tens 500 columnes i un milió de files. Ja no pots mirar-t'ho.

## 4. Deixem que la màquina les escrigui sola

Aquí comença el machine learning. En lloc d'escriure les regles nosaltres, **donem les
dades a un programa i que en dedueixi les regles**.

Primer separem les dades en dos grups: un per ensenyar-li i un per examinar-lo. Això és
important: si l'examinem amb les mateixes flors que li hem ensenyat, no sabrem si ha
après o s'ho ha memoritzat.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X = dades[["sepal_llarg", "sepal_ample", "petal_llarg", "petal_ample"]]
y = dades["especie"]

X_entrena, X_examen, y_entrena, y_examen = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Per ensenyar-li: {len(X_entrena)} flors")
print(f"Per examinar-lo: {len(X_examen)} flors")

arbre = DecisionTreeClassifier(max_depth=3, random_state=42)
arbre.fit(X_entrena, y_entrena)

print(f"\nPrecisió a l'examen: {arbre.score(X_examen, y_examen):.1%}")

## 5. I ara la part bona: mirem què ha escrit

Un arbre de decisió **no és una caixa negra**. Es pot dibuixar. I quan el dibuixes, veus
que el que ha après són exactament els teus `if` i `else`, però trobats sol a partir de
les dades.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(14, 7))
plot_tree(
    arbre,
    feature_names=list(X.columns),
    class_names=list(arbre.classes_),
    filled=True,
    rounded=True,
    fontsize=10,
)
plt.title("Els condicionals que ha escrit la màquina")
plt.show()

Llegeix el node de dalt de tot. Diu alguna cosa com *petal_llarg <= 2.45*.

**És el teu 2.5.** No l'hi has dit tu: l'ha trobat ell mirant les 105 flors d'entrenament.

I el següent nivell fa el mateix amb l'amplada del pètal, que és el teu 1.75.

Això és tot el misteri. Un arbre de decisió és un `if/elif/else` que algú ha escrit
llegint dades en comptes de mirant un gràfic.

## 6. I si en provem uns quants?

L'arbre és un model entre molts. Aquí n'hi ha cinc, i cadascun té la seva manera de
decidir. Tots s'entrenen igual: `.fit()` per aprendre i `.score()` per examinar-se.

Tres línies per model. Aquesta és la feina que fa scikit-learn per tu.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

models = {
    "Regressió logística": LogisticRegression(max_iter=200, random_state=42),
    "Arbre de decisió": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Bosc aleatori": RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42),
    "SVM": SVC(kernel="rbf", random_state=42),
    "k veïns més propers": KNeighborsClassifier(n_neighbors=5),
}

resultats = []
for nom, model in models.items():
    model.fit(X_entrena, y_entrena)
    resultats.append({"Model": nom, "Precisió": f"{model.score(X_examen, y_examen):.1%}"})

pd.DataFrame(resultats)

Val la pena parar-se en una cosa: **el bosc aleatori surt pitjor que l'arbre sol**, tot i
que un bosc són cent arbres votant.

No és cap error. Iris és un conjunt minúscul i fàcil, i aquí un model més complicat no
ajuda: només afegeix soroll. Això passa sovint i és una de les lliçons del curs, **el
model més potent no és el que guanya sempre**.

Fixa't també que canviar el `random_state` mou aquests números. Amb 45 flors d'examen,
una flor amunt o avall ja són dos punts de precisió.

## 7. On s'equivoca

L'arbre encerta un 97,8 %, o sigui que falla **una sola flor de les 45**. Però **no totes
les errades són iguals**, i convé saber quina ha confós.

La matriu de confusió ho ensenya: les files són l'espècie real i les columnes la que ha
dit el model. Tot el que no és a la diagonal és una errada.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    arbre, X_examen, y_examen, cmap="Blues", colorbar=False
)
plt.title("On s'equivoca l'arbre")
plt.show()

## Resum del que acabes de veure

- Has classificat flors **a mà**, amb els condicionals d'ahir, i has encertat un 96 %.
- Has vist que **una màquina troba aquestes mateixes regles sola**, i que es poden
  dibuixar i llegir.
- Has entrenat **cinc models diferents** amb tres línies cadascun.
- Has vist que **el model més complicat no sempre guanya**.
- Has mirat **on s'equivoquen**, que és tan important com quant encerten.

### I què farem durant el curs, doncs?

Precisament el que aquest quadern s'ha saltat:

- **Per què** cal separar entrenament i examen, i què passa si no ho fas.
- **Com decideix** cada model, i per què un va bé aquí i malament allà.
- **Què fer quan les dades són lletges**: valors que falten, columnes de text, escales
  diferents. Iris és un dataset de joguina; els de veritat no vénen així.
- **Quan no et pots fiar** d'un 95 %.

Res d'això s'aprèn mirant. Cap al gener, aquest quadern el sabràs escriure tu.